## Notebook for merging prerpcosesed datasets
### Step 0: Set up paths

In [ ]:
# ==========================================
# STEP 0 — Paths & load cleaned datasets
# ==========================================
from pathlib import Path
import pandas as pd

ROOT    = Path("/home/aidan/IMU_LM_Data")
CLEANED = ROOT / "data" / "cleaned_premerge"
MERGED  = ROOT / "data" / "merged_dataset"

print("CLEANED dir :", CLEANED)
print("MERGED  dir :", MERGED)

# Cleaned parquet files we expect (only load ones that actually exist)
DATASETS = {
    "opportunity": "opportunity_clean_data.parquet",
    "pamap2":      "pamap2_clean_data.parquet",
    "recofit":     "recofit_clean_data.parquet",
    "samosa":      "samosa_clean_data.parquet",
    "ut_watch":    "ut_watch_clean_data.parquet",
    "wear":        "wear_clean_data.parquet",
}

loaded = {}
for ds_name, fname in DATASETS.items():
    path = CLEANED / fname
    if not path.exists():
        print(f"[WARN] Missing cleaned file for {ds_name}: {path}")
        continue

    df = pd.read_parquet(path)
    print(f"[OK] Loaded {ds_name:10s}: {len(df):,} rows")
    loaded[ds_name] = df

if not loaded:
    raise SystemExit("No cleaned datasets loaded — check CLEANED path / filenames.")

print("\nLoaded datasets:", list(loaded.keys()))



CLEANED dir : /home/aidan/IMU_LM_Data/data/cleaned_premerge
MERGED  dir : /home/aidan/IMU_LM_Data/data/merged_dataset
[OK] Loaded opportunity: 1,215,455 rows
[OK] Loaded pamap2    : 1,271,148 rows
[OK] Loaded recofit   : 7,751,906 rows
[OK] Loaded samosa    : 1,205,141 rows
[OK] Loaded ut_watch  : 8,146,709 rows
[OK] Loaded wear      : 3,466,400 rows

Loaded datasets: ['opportunity', 'pamap2', 'recofit', 'samosa', 'ut_watch', 'wear']


### Step 1: Create Master Dataset

In [2]:
# ==========================================
# STEP 1 — Merge, sanity checks, save
# ==========================================

# 1) Concatenate (simple stack)
unified = pd.concat(
    [loaded[k] for k in sorted(loaded.keys())],
    ignore_index=True
)

# 2) Sort by core index columns
unified = unified.sort_values(
    ["dataset", "subject_id", "session_id", "timestamp_ns"]
).reset_index(drop=True)

print(f"\nUNIFIED rows: {len(unified):,}")

# 3) Basic stats
print("\nRows per dataset:")
print(unified["dataset"].value_counts())

print("\nSubjects per dataset:")
print(unified.groupby("dataset")["subject_id"].nunique())

print("\nSessions per dataset:")
print(unified.groupby("dataset")["session_id"].nunique())

# 4) Required-not-null coverage (hard-coded from schema)
REQ_NOT_NULL = [
    "dataset", "subject_id", "session_id", "timestamp_ns",
    "acc_x", "acc_y", "acc_z",
    "global_activity_id", "global_activity_label",
    "dataset_activity_id", "dataset_activity_label",
]
pct_req = unified[REQ_NOT_NULL].notnull().all(axis=1).mean() * 100
print(f"\nRows meeting required-not-null contract: {pct_req:.2f}%")

# 5) Global mapping coverage + label distribution
UNKNOWN_ID = 9000
cov = (unified["global_activity_id"] != UNKNOWN_ID).mean() * 100
print(f"Global activity mapping coverage: {cov:.1f}% (unknown={UNKNOWN_ID})")

print("\nTop-20 canonical labels (global_activity_label):")
print(unified["global_activity_label"].value_counts().head(20))

# 6) Quick 1–1 check: global ID → label
pairs = unified[["global_activity_id", "global_activity_label"]].drop_duplicates()
id_to_lbl_counts = pairs.groupby("global_activity_id")["global_activity_label"].nunique()
print("\nGlobal id→label one-to-one:", bool((id_to_lbl_counts == 1).all()))

# 7) Save unified parquet
MERGED.mkdir(parents=True, exist_ok=True)
out_path = MERGED / "unified_dataset.parquet"
unified.to_parquet(out_path, index=False)
print("\nSaved unified dataset to:", out_path)
print("Done.")


UNIFIED rows: 23,056,759

Rows per dataset:
dataset
ut_watch         8146709
recofit          7751906
wear             3466400
pamap2           1271148
opportunity++    1215455
samosa           1205141
Name: count, dtype: int64

Subjects per dataset:
dataset
opportunity++     4
pamap2            9
recofit          94
samosa           20
ut_watch         20
wear             24
Name: subject_id, dtype: int64

Sessions per dataset:
dataset
opportunity++      6
pamap2             2
recofit           73
samosa           125
ut_watch           4
wear               1
Name: session_id, dtype: int64

Rows meeting required-not-null contract: 100.00%
Global activity mapping coverage: 92.6% (unknown=9000)

Top-20 canonical labels (global_activity_label):
global_activity_label
rest_inactive            7224537
other                    1805574
exercise_upper           1765088
stretching               1695420
device_offbody           1301735
exercise_core            1165114
walk                     1